In [5]:
# 9_group_averages.ipynb
#
# Computes group-level baselines for each (unit_id, employment group) from the
# synthetic population parquet.  Runs once for local clusters and once for
# national clusters, producing four CSVs in data/9_group_averages/:
#
#   local_group_baselines.csv      — one row per (unit_id, group) with
#   local_group_distributions.csv    continuous means, categorical modes, and
#                                    full categorical breakdowns (local scope).
#
#   national_group_baselines.csv   — same schema, national cluster scope.
#   national_group_distributions.csv
#
# These allow steps 10/11 (label_clusters) to tell the LLM how each cluster
# differs from its group average, producing more insightful descriptions.

import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config_paths import DATA_FOLDER, USE_FOUR_LA_SUBSET, FOUR_LA_CODES
import data_pipeline.config_variables as _cv
import data_pipeline.config_cluster   as _cc
_cv.reload_config_variables()
importlib.reload(_cc)

import pandas as pd
import numpy  as np
from pathlib import Path
from tqdm    import tqdm

import data_pipeline.helpers.cluster as cf
importlib.reload(cf)

from data_pipeline.config_variables import (
    SUMMARY_VARS, VARIABLE_MAP,
    CATEGORICAL_VARS, CATEGORY_MAPS, CONTINUOUS_VARS,
)
from data_pipeline.config_cluster import WAVE, GROUPS

# ── Config ────────────────────────────────────────────────────────────────────
SYNPOP_PARQUET = f"../{DATA_FOLDER}/6_synthetic_population/synthetic_population.parquet"
OUTPUT_DIR     = Path(f"../{DATA_FOLDER}/9_group_averages")
GEO_CSV        = Path(f"../{DATA_FOLDER}/0_raw/admin_geography_mappings.csv")

CLUSTER_CSVS = {
    "local":    Path(f"../{DATA_FOLDER}/7_cluster_local_level/LA_london_clusters.csv"),
    "national": Path(f"../{DATA_FOLDER}/8_cluster_national_level/LA_london_national_clusters.csv"),
}

UNIT_FILTER = list(FOUR_LA_CODES) if USE_FOUR_LA_SUBSET else "london"
LEVEL_COL   = "ladcd"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not os.path.exists(SYNPOP_PARQUET):
    raise FileNotFoundError(f"{SYNPOP_PARQUET} not found — run 6_synthetic_population first.")

# ── LA name lookup ────────────────────────────────────────────────────────────
_la_name_map = {}
if GEO_CSV.exists():
    _geo = pd.read_csv(GEO_CSV, encoding='latin-1', usecols=['ladcd', 'ladnm']).drop_duplicates('ladcd')
    _la_name_map = _geo.set_index('ladcd')['ladnm'].to_dict()

# ── Jbstat group assignment (same logic as clustering notebooks) ──────────────
import pyarrow.parquet as pq
parquet_cols = set(pq.read_schema(SYNPOP_PARQUET).names)

_JBSTAT_LOOKUP = {
    "Employed":   f"{WAVE}_jbstat_1",
    "Unemployed": f"{WAVE}_jbstat_3",
    "Retired":    f"{WAVE}_jbstat_4",
    "On leave":   f"{WAVE}_jbstat_5",
    "Student":    f"{WAVE}_jbstat_7",
    "Inactive":   f"{WAVE}_jbstat_8",
}
GROUP_COL = {
    gname: (
        _JBSTAT_LOOKUP[gname]
        if _JBSTAT_LOOKUP.get(gname) in parquet_cols
        else (None if _JBSTAT_LOOKUP.get(gname) is None else "MISSING")
    )
    for gname in GROUPS
}


def _resolve_unit_ids(cluster_csv: Path) -> list[str]:
    """Read unit_ids from a cluster CSV, filtered by UNIT_FILTER."""
    df = pd.read_csv(cluster_csv)
    ids = sorted(df['unit_id'].unique().tolist())
    if UNIT_FILTER is None:
        return ids
    if UNIT_FILTER == "london":
        return [u for u in ids if str(u).startswith("E09")]
    if isinstance(UNIT_FILTER, list):
        allow = set(UNIT_FILTER)
        return [u for u in ids if u in allow]
    raise ValueError("UNIT_FILTER must be None, 'london', or a list of LA codes")


def compute_group_averages(unit_ids: list[str], label: str):
    """Compute baselines and distributions for *unit_ids*, return (df_baselines, df_dist)."""
    baseline_rows: list[dict] = []
    dist_rows: list[dict] = []

    for unit_id in tqdm(unit_ids, desc=f"Group averages ({label})"):
        df_unit = pd.read_parquet(
            SYNPOP_PARQUET,
            filters=[(LEVEL_COL, '==', unit_id)],
        )
        if df_unit.empty:
            continue

        la_name = _la_name_map.get(unit_id, unit_id)

        # ── Total population baseline (all groups combined) ───────────
        total_row = cf.build_dna_row(
            "Total (baseline)", df_unit, WAVE,
            SUMMARY_VARS, VARIABLE_MAP, CATEGORICAL_VARS, CATEGORY_MAPS,
            continuous_vars=CONTINUOUS_VARS,
        )
        total_row['unit_id'] = unit_id
        total_row['la_name'] = la_name
        total_row['group']   = 'Total'
        baseline_rows.append(total_row)

        for base_code in SUMMARY_VARS:
            if base_code not in CATEGORICAL_VARS:
                continue
            cat_map = CATEGORY_MAPS.get(base_code)
            if not cat_map:
                continue
            col_name = f"{WAVE}_{base_code}"
            if col_name not in df_unit.columns:
                continue
            series = pd.to_numeric(df_unit[col_name], errors='coerce').dropna()
            total  = len(series)
            if total == 0:
                continue
            var_label = VARIABLE_MAP.get(base_code, base_code)
            counts = series.value_counts()
            for cat_code, count in counts.items():
                dist_rows.append({
                    'unit_id':       unit_id,
                    'la_name':       la_name,
                    'group':         'Total',
                    'variable':      var_label,
                    'category_code': cat_code,
                    'category':      cat_map.get(cat_code, str(cat_code)),
                    'count':         int(count),
                    'pct':           round(count / total * 100, 1),
                })

        # ── Per-group baselines ───────────────────────────────────────
        assigned = pd.Series(False, index=df_unit.index)

        for gname in GROUPS:
            col = GROUP_COL.get(gname)
            if col == "MISSING":
                continue
            if col is not None and col in df_unit.columns:
                mask = df_unit[col] == 1.0
            elif col is None:
                mask = ~assigned
            else:
                continue

            group_df = df_unit[mask]
            assigned |= mask
            if group_df.empty:
                continue

            row = cf.build_dna_row(
                f"{gname} (baseline)", group_df, WAVE,
                SUMMARY_VARS, VARIABLE_MAP, CATEGORICAL_VARS, CATEGORY_MAPS,
                continuous_vars=CONTINUOUS_VARS,
            )
            row['unit_id'] = unit_id
            row['la_name'] = la_name
            row['group']   = gname
            baseline_rows.append(row)

            for base_code in SUMMARY_VARS:
                if base_code not in CATEGORICAL_VARS:
                    continue
                cat_map = CATEGORY_MAPS.get(base_code)
                if not cat_map:
                    continue
                col_name = f"{WAVE}_{base_code}"
                if col_name not in group_df.columns:
                    continue

                series = pd.to_numeric(group_df[col_name], errors='coerce').dropna()
                total  = len(series)
                if total == 0:
                    continue

                var_label = VARIABLE_MAP.get(base_code, base_code)
                counts = series.value_counts()
                for cat_code, count in counts.items():
                    dist_rows.append({
                        'unit_id':       unit_id,
                        'la_name':       la_name,
                        'group':         gname,
                        'variable':      var_label,
                        'category_code': cat_code,
                        'category':      cat_map.get(cat_code, str(cat_code)),
                        'count':         int(count),
                        'pct':           round(count / total * 100, 1),
                    })

        del df_unit
        gc.collect()

    return pd.DataFrame(baseline_rows), pd.DataFrame(dist_rows)


# ── Run for each cluster level ────────────────────────────────────────────────
for level, csv_path in CLUSTER_CSVS.items():
    if not csv_path.exists():
        print(f"SKIP {level}: {csv_path} not found")
        continue

    uids = _resolve_unit_ids(csv_path)
    if not uids:
        print(f"SKIP {level}: no units after UNIT_FILTER")
        continue

    print(f"\n{'='*60}")
    print(f"Computing {level} group averages for {len(uids)} units "
          f"[USE_FOUR_LA_SUBSET={USE_FOUR_LA_SUBSET}]")

    df_bl, df_di = compute_group_averages(uids, level)

    bl_path = OUTPUT_DIR / f"{level}_group_baselines.csv"
    di_path = OUTPUT_DIR / f"{level}_group_distributions.csv"

    df_bl.to_csv(bl_path, index=False)
    print(f"Saved {len(df_bl)} baseline rows to {bl_path.name}")

    df_di.to_csv(di_path, index=False)
    print(f"Saved {len(df_di)} distribution rows to {di_path.name}")

    display(df_bl[['unit_id', 'group', 'size']].head(8))



Computing local group averages for 4 units [USE_FOUR_LA_SUBSET=True]


Group averages (local): 100%|██████████| 4/4 [00:00<00:00,  6.16it/s]


Saved 24 baseline rows to local_group_baselines.csv
Saved 984 distribution rows to local_group_distributions.csv


,unit_id,group,size
0,E09000018,Employed,84192
1,E09000018,Retired,22903
2,E09000018,Unemployed,8038
3,E09000018,Student,3142
4,E09000018,On leave,4720
5,E09000018,Inactive,5857
6,E09000019,Employed,84796
7,E09000019,Retired,17335



Computing national group averages for 4 units [USE_FOUR_LA_SUBSET=True]


Group averages (national): 100%|██████████| 4/4 [00:00<00:00,  9.85it/s]

Saved 24 baseline rows to national_group_baselines.csv
Saved 984 distribution rows to national_group_distributions.csv


,unit_id,group,size
0,E09000018,Employed,84192
1,E09000018,Retired,22903
2,E09000018,Unemployed,8038
3,E09000018,Student,3142
4,E09000018,On leave,4720
5,E09000018,Inactive,5857
6,E09000019,Employed,84796
7,E09000019,Retired,17335
